## 1.  Cargar el Dataset

In [4]:
# Instalación de librerías necesarias (solo si no están disponibles)
# !pip install pandas scikit-learn openpyxl --quiet

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías importadas correctamente')

✅ Librerías importadas correctamente


In [7]:
# ─────────────────────────────────────────────────────────────
# OPCIÓN A: Subir el archivo manualmente en Google Colab
# ─────────────────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()  # Selecciona tu archivo CSV
ARCHIVO = list(uploaded.keys())[0]

ENCODING = 'latin1' # Se cambia la codificación a 'latin1'
df = pd.read_csv(ARCHIVO, encoding=ENCODING)

print(f'✅ Dataset cargado exitosamente')
print(f'   Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')

Saving sales_data_sample.csv to sales_data_sample (2).csv
✅ Dataset cargado exitosamente
   Filas: 2,823  |  Columnas: 25


In [8]:
# Vista rápida del dataset
print('── Primeras filas ──────────────────────────────────────')
display(df.head())

print('\n── Tipos de datos ──────────────────────────────────────')
print(df.dtypes)

print('\n── Valores nulos por columna ───────────────────────────')
nulos = df.isnull().sum()
print(nulos[nulos > 0])

print('\n── Estadísticas descriptivas (columnas numéricas) ──────')
display(df.describe())

── Primeras filas ──────────────────────────────────────


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium



── Tipos de datos ──────────────────────────────────────
ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID               int64
PRODUCTLINE          object
MSRP                  int64
PRODUCTCODE          object
CUSTOMERNAME         object
PHONE                object
ADDRESSLINE1         object
ADDRESSLINE2         object
CITY                 object
STATE                object
POSTALCODE           object
COUNTRY              object
TERRITORY            object
CONTACTLASTNAME      object
CONTACTFIRSTNAME     object
DEALSIZE             object
dtype: object

── Valores nulos por columna ───────────────────────────
ADDRESSLINE2    2521
STATE           1486
POSTALCODE        76
TERRITORY       1074
dtype: int64

── Estadísticas descriptivas (columnas numéricas) ──────


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,QTR_ID,MONTH_ID,YEAR_ID,MSRP
count,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.00000,2823.000000
mean,10258.725115,35.092809,83.658544,6.466171,3553.889072,2.717676,7.092455,2003.81509,100.715551
std,92.085478,9.741443,20.174277,4.225841,1841.865106,1.203878,3.656633,0.69967,40.187912
min,10100.000000,6.000000,26.880000,1.000000,482.130000,1.000000,1.000000,2003.00000,33.000000
25%,10180.000000,27.000000,68.860000,3.000000,2203.430000,2.000000,4.000000,2003.00000,68.000000
50%,10262.000000,35.000000,95.700000,6.000000,3184.800000,3.000000,8.000000,2004.00000,99.000000
75%,10333.500000,43.000000,100.000000,9.000000,4508.000000,4.000000,11.000000,2004.00000,124.000000
max,10425.000000,97.000000,100.000000,18.000000,14082.800000,4.000000,12.000000,2005.00000,214.000000


---
## 2. Normalización del Dataset



In [9]:
# Trabajamos sobre una copia para no modificar el original
df_clean = df.copy()
print('✅ Copia de trabajo creada')

✅ Copia de trabajo creada


In [10]:
# ── PASO 1: Tratamiento de valores nulos ────────────────────

# Columnas numéricas: rellenar con la mediana
cols_numericas = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in cols_numericas:
    if df_clean[col].isnull().sum() > 0:
        mediana = df_clean[col].median()
        df_clean[col].fillna(mediana, inplace=True)
        print(f'   [{col}] → nulos reemplazados con mediana ({mediana})')

# Columnas de texto: rellenar con 'Desconocido'
cols_texto = df_clean.select_dtypes(include=['object']).columns.tolist()
for col in cols_texto:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna('Desconocido', inplace=True)
        print(f'   [{col}] → nulos reemplazados con "Desconocido"')

total_nulos = df_clean.isnull().sum().sum()
print(f'\n✅ Paso 1 completo | Valores nulos restantes: {total_nulos}')

   [ADDRESSLINE2] → nulos reemplazados con "Desconocido"
   [STATE] → nulos reemplazados con "Desconocido"
   [POSTALCODE] → nulos reemplazados con "Desconocido"
   [TERRITORY] → nulos reemplazados con "Desconocido"

✅ Paso 1 completo | Valores nulos restantes: 0


In [11]:
# ── PASO 2: Conversión de tipos de datos ────────────────────

# Convertir ORDERDATE a datetime
if 'ORDERDATE' in df_clean.columns:
    df_clean['ORDERDATE'] = pd.to_datetime(df_clean['ORDERDATE'], errors='coerce')
    print('   [ORDERDATE] → convertida a datetime')

# Limpiar espacios en columnas de texto
for col in cols_texto:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print('✅ Paso 2 completo | Tipos de datos ajustados')
print(df_clean.dtypes)

   [ORDERDATE] → convertida a datetime
✅ Paso 2 completo | Tipos de datos ajustados
ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID               int64
PRODUCTLINE          object
MSRP                  int64
PRODUCTCODE          object
CUSTOMERNAME         object
PHONE                object
ADDRESSLINE1         object
ADDRESSLINE2         object
CITY                 object
STATE                object
POSTALCODE           object
COUNTRY              object
TERRITORY            object
CONTACTLASTNAME      object
CONTACTFIRSTNAME     object
DEALSIZE             object
dtype: object


In [12]:
# ── PASO 3: Normalización Min-Max (escala 0 a 1) ────────────
#
# Ideal para algoritmos basados en distancias (KNN, redes neuronales).
# Conserva la distribución original.

# Columnas a normalizar (excluimos IDs y columnas de año/mes/trimestre)
EXCLUIR = ['ORDERNUMBER', 'ORDERLINENUMBER', 'QTR_ID', 'MONTH_ID', 'YEAR_ID']
cols_normalizar = [
    c for c in cols_numericas
    if c not in EXCLUIR
]

print(f'Columnas a normalizar con Min-Max (se sobrescribirán): {cols_normalizar}\n')

scaler_minmax = MinMaxScaler()

for col in cols_normalizar:
    df_clean[col] = scaler_minmax.fit_transform(df_clean[[col]]) # Sobrescribir columna original
    print(f'   [{col}] → rango: [{df_clean[col].min():.4f}, {df_clean[col].max():.4f}]')

print('\n✅ Paso 3 completo | Normalización Min-Max aplicada (columnas sobrescritas)')

Columnas a normalizar con Min-Max (se sobrescribirán): ['QUANTITYORDERED', 'PRICEEACH', 'SALES', 'MSRP']

   [QUANTITYORDERED] → rango: [0.0000, 1.0000]
   [PRICEEACH] → rango: [0.0000, 1.0000]
   [SALES] → rango: [0.0000, 1.0000]
   [MSRP] → rango: [0.0000, 1.0000]

✅ Paso 3 completo | Normalización Min-Max aplicada (columnas sobrescritas)


In [13]:
# ── PASO 4: Estandarización Z-Score (media=0, std=1) ────────
#
# Ideal para modelos lineales, PCA, SVM.
# Maneja mejor los outliers que Min-Max.

print(f'Columnas a estandarizar con Z-Score (se sobrescribirán): {cols_normalizar}\n')

scaler_std = StandardScaler()

for col in cols_normalizar:
    df_clean[col] = scaler_std.fit_transform(df_clean[[col]]) # Sobrescribir columna original
    print(f'   [{col}] → media: {df_clean[col].mean():.4f}  std: {df_clean[col].std():.4f}')

print('\n✅ Paso 4 completo | Estandarización Z-Score aplicada (columnas sobrescritas)')

Columnas a estandarizar con Z-Score (se sobrescribirán): ['QUANTITYORDERED', 'PRICEEACH', 'SALES', 'MSRP']

   [QUANTITYORDERED] → media: 0.0000  std: 1.0002
   [PRICEEACH] → media: 0.0000  std: 1.0002
   [SALES] → media: 0.0000  std: 1.0002
   [MSRP] → media: 0.0000  std: 1.0002

✅ Paso 4 completo | Estandarización Z-Score aplicada (columnas sobrescritas)


In [14]:
# ── Resumen final del dataset procesado ─────────────────────

print('── Dataset limpio y normalizado (columnas originales sobrescritas) ──')
print(f'   Filas:    {df_clean.shape[0]:,}')
print(f'   Columnas: {df_clean.shape[1]}')
print(f'   Nulos:    {df_clean.isnull().sum().sum()}')

print('\n── Vista previa (columnas numéricas ahora Z-score estandarizadas) ──')
# Mostrar columnas originales ahora normalizadas
cols_vista = ['ORDERNUMBER', 'SALES', 'QUANTITYORDERED', 'PRICEEACH', 'MSRP']
display(df_clean[cols_vista].head(10))

── Dataset limpio y normalizado (columnas originales sobrescritas) ──
   Filas:    2,823
   Columnas: 25
   Nulos:    0

── Vista previa (columnas numéricas ahora Z-score estandarizadas) ──


,ORDERNUMBER,SALES,QUANTITYORDERED,PRICEEACH,MSRP
0,10107,-0.370825,-0.522891,0.596978,-0.142246
1,10121,-0.427897,-0.112201,-0.114450,-0.142246
2,10134,0.179443,0.606505,0.549384,-0.142246
3,10145,0.104701,1.017195,-0.019759,-0.142246
4,10159,0.896740,1.427884,0.810158,-0.142246
5,10168,-0.040254,0.093143,0.644571,-0.142246
6,10180,-0.573498,-0.625563,0.122527,-0.142246
7,10188,1.063475,1.325212,0.810158,-0.142246
8,10201,-0.752278,-1.344270,0.739263,-0.142246
9,10211,0.626949,0.606505,0.810158,-0.142246


---
## 3. Guardar el Dataset Limpio

In [15]:
# ── Guardar el dataset completo (original + columnas normalizadas)
SALIDA_COMPLETO = 'sales_data_normalizado.csv'

df_clean.to_csv(SALIDA_COMPLETO, index=False, encoding='utf-8')
print(f'✅ Dataset completo guardado → {SALIDA_COMPLETO}')
print(f'   Tamaño: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas')

✅ Dataset completo guardado → sales_data_normalizado.csv
   Tamaño: 2,823 filas × 25 columnas
